# Quantum Vision Transformers for Tyre Defect Detection
## A Comparative Benchmark: CNN vs. ViT vs. QCNN vs. QViT on the TyreNet Dataset

**Target organization:** Apollo Tyres (Industrial Quality Control)  
**Dataset:** TyreNet (Mendeley Data) — ~1,700 high-resolution images of *good* vs. *defective* tyres.  
**Frameworks:** PyTorch (classical) + PennyLane (variational quantum circuits).

### Research hypothesis

> Hybrid **Quantum Vision Transformers (QViTs)** offer superior **parameter efficiency** and a stronger **inductive bias for global feature extraction** than classical CNNs and ViTs, while requiring far fewer trainable parameters.

This is grounded in two recent results:
* **Cherrat et al. (2024)** — show that *Quantum Self-Attention* (QSA) admits $O(n)$ parameter scaling versus the $O(n^2)$ of classical self-attention.
* **Boucher et al. (2025)** — discuss the inductive bias of variational quantum attention for capturing global structure.

### What this notebook does

1. Spins up the Colab environment with `pennylane-lightning[gpu]`.
2. Pulls the TyreNet dataset and builds a *dual-resolution* PyTorch Dataset (classical $224\times224$ view + a quantum-ready per-patch reduced view).
3. Walks through the **Hybrid Quantum Attention Head** in detail.
4. Trains all four architectures — `ClassicalCNN`, `ViT-Tiny`, `QCNN`, `QViT` — and optionally distills a `QViT` student from a classical `ViT-Tiny` teacher.
5. Reports the three required plots: **Accuracy vs. #Params**, **Accuracy vs. Epochs**, and **FLOPs vs. Quantum Gate Count**, plus a written analysis.

## 0. Colab environment setup

If you are running this notebook in Google Colab, enable a **GPU runtime** first (*Runtime → Change runtime type → T4/A100*).

The cell below clones our repository, installs PyTorch / PennyLane / `pennylane-lightning[gpu]`, and adds the project's `src/` package to `PYTHONPATH`.

In [ ]:
# --- One-time Colab setup ---
import os, sys, subprocess, importlib

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/khushib004/QViT_Experiment.git'
BRANCH   = 'claude/benchmark-vision-defect-detection-n1erk'
REPO_DIR = '/content/QViT_Experiment' if IN_COLAB else os.path.abspath('..')

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR])
    subprocess.check_call(['pip', 'install', '-q',
                            'pennylane>=0.35',
                            'pennylane-lightning[gpu]',
                            'scikit-learn', 'matplotlib', 'tqdm', 'pillow'])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Repo dir:', REPO_DIR)
print('In Colab :', IN_COLAB)

In [ ]:
import torch, pennylane as qml
print('PyTorch       :', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
print('PennyLane     :', qml.__version__)

# Pick the quantum device with the fastest backend that's actually available.
def pick_qdevice():
    for cand in ['lightning.gpu', 'lightning.qubit', 'default.qubit']:
        try:
            qml.device(cand, wires=2)
            return cand
        except Exception:
            continue
    return 'default.qubit'

QDEVICE = pick_qdevice()
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using quantum :', QDEVICE)
print('Using torch on:', DEVICE)

## 1. Dataset — TyreNet

TyreNet is hosted on Mendeley Data. The simplest way to use it in Colab is one of:

* **(a)** Upload it as a ZIP and unzip it into `/content/data/tyrenet`
* **(b)** Mount Google Drive and point to a copy you keep there
* **(c)** Download programmatically with `kagglehub` / `wget` if you have a direct link.

All routes converge on a folder layout that the `TyreNetDataset` understands:

```
data/tyrenet/
  good/
      img_0001.jpg
      ...
  defective/
      img_0001.jpg
      ...
```

The next cell creates a small **synthetic mock** so the rest of the notebook can be executed end-to-end without the real dataset (useful while developing). Replace it with the real path before running the actual benchmark.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

DATA_ROOT = Path('data/tyrenet')
USE_MOCK  = not (DATA_ROOT / 'good').exists()

if USE_MOCK:
    print('[mock] No real TyreNet found — generating a tiny synthetic dataset.')
    rng = np.random.default_rng(0)
    for cls, n in [('good', 60), ('defective', 60)]:
        d = DATA_ROOT / cls
        d.mkdir(parents=True, exist_ok=True)
        for i in range(n):
            base = rng.integers(40, 160, size=(224, 224, 3), dtype=np.uint8)
            if cls == 'defective':
                # add a 'crack' streak so the classifier has signal
                y0, x0 = rng.integers(20, 200), rng.integers(20, 200)
                base[y0:y0+5, x0:x0+80, :] = 0
            Image.fromarray(base).save(d / f'{cls}_{i:04d}.png')
    print('[mock] Wrote', sum(1 for _ in DATA_ROOT.rglob('*.png')), 'mock images.')
else:
    print('Using real dataset at', DATA_ROOT.resolve())

In [ ]:
from src.data import TyreNetDataset

N_QUBITS   = 4    # 4-8 recommended; lightning.gpu scales reasonably to ~12
PATCH_SIZE = 32   # 224/32 = 7x7 = 49 patches in the quantum view

train_ds = TyreNetDataset(root=str(DATA_ROOT), split='train', n_qubits=N_QUBITS, patch_size=PATCH_SIZE, reducer='conv')
val_ds   = TyreNetDataset(root=str(DATA_ROOT), split='val',   n_qubits=N_QUBITS, patch_size=PATCH_SIZE, reducer='conv')
test_ds  = TyreNetDataset(root=str(DATA_ROOT), split='test',  n_qubits=N_QUBITS, patch_size=PATCH_SIZE, reducer='conv')

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print('Num quantum patches per image:', train_ds.num_patches)
sample = train_ds[0]
print('image_classical:', tuple(sample["image_classical"].shape))
print('image_quantum :', tuple(sample["image_quantum"].shape))
print('label         :', sample["label"].item())

In [ ]:
import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_pil_image

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    s = train_ds[i]
    img = s['image_classical']
    # de-normalise approximately for visualisation
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    ax.imshow(to_pil_image(img))
    ax.set_title('defective' if s['label'].item() == 1 else 'good')
    ax.axis('off')
plt.suptitle('TyreNet samples (classical 224x224 view)')
plt.tight_layout(); plt.show()

### 1.1 Dual-resolution: why and how

Classical models consume the full $3\times224\times224$ tensor. A quantum register cannot — current `lightning.gpu` simulators are practical up to ~20 qubits, so we must **compress each image patch to `n_qubits` features** before angle-encoding it.

Our `TyreNetDataset` provides two reducers (see `src/data/tyrenet_dataset.py`):

* **`StridedConvReducer`** — a single frozen `Conv2d(3, n_qubits, kernel=patch, stride=patch)` with orthogonal init, followed by `tanh` to keep features in $[-1, 1]$ (the regime `AngleEmbedding` likes).
* **`PCAReducer`** — fits PCA on flattened patches from the training split, then `tanh`-squashes the projection.

The conv reducer is faster and differentiable-compatible; PCA is data-adaptive and useful for ablations.

In [ ]:
qv = sample['image_quantum']  # (P, n_qubits)
print('Quantum view shape:', tuple(qv.shape), 'range:', float(qv.min()), float(qv.max()))

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
im = ax.imshow(qv.T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xlabel('Patch index'); ax.set_ylabel('Qubit feature')
ax.set_title('Quantum-ready per-patch features (input to AngleEmbedding)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 2. The Hybrid Quantum Attention Head — theory

Classical Multi-Head Self-Attention computes

$$ \mathrm{Attn}(X) = \mathrm{softmax}\!\Big(\frac{(XW_Q)(XW_K)^\top}{\sqrt{d}}\Big)\, XW_V $$

with three learned $D\times D$ projection matrices per head — i.e. **$3D^2$** parameters per head ($O(n^2)$ in the embedding dim).

Our **Quantum Self-Attention** replaces each of $W_Q, W_K, W_V$ with a *Variational Quantum Circuit (VQC)* applied **per-token**:

1. **State preparation:** `AngleEmbedding` maps the token's reduced features (size = `n_qubits`) to single-qubit $R_Y$ rotations.
2. **Variational ansatz:** `BasicEntanglerLayers` interleaves $R_X$ rotations with a ring of CNOTs (hardware-efficient, $O(n_{qubits})$ parameters per layer).
3. **Readout:** $\langle Z_i \rangle$ on each wire yields an `n_qubits`-dim per-token vector.

The variational ansatz contains only **$n_{layers} \times n_{qubits}$** trainable parameters per Q/K/V circuit — the $O(n)$ scaling reported by Cherrat et al. (2024).

The softmax stays classical: there is no efficient unitary equivalent, so we keep it on the host.

In [ ]:
# Visualise the actual circuit shape using PennyLane's drawer.
import pennylane as qml
from src.models.quantum_attention import _build_qnode  # internal helper, fine for diagnostics

n_qubits, n_layers = 4, 2
qnode = _build_qnode(n_qubits=n_qubits, n_layers=n_layers, device_name=QDEVICE)
import torch
dummy_in     = torch.zeros(n_qubits)
dummy_weights = torch.zeros(n_layers, n_qubits)
print(qml.draw(qnode)(dummy_in, dummy_weights))

In [ ]:
# Parameter accounting comparison: classical MHA vs. our QSA, for the same embed_dim.
import torch.nn as nn
from src.models.quantum_attention import HybridQuantumMultiHeadAttention

embed_dim = 96
n_heads, n_qubits, n_layers = 2, 4, 2

classical_mha = nn.MultiheadAttention(embed_dim, num_heads=n_heads, batch_first=True)
quantum_mha   = HybridQuantumMultiHeadAttention(
    embed_dim=embed_dim, n_heads=n_heads,
    n_qubits=n_qubits, n_layers=n_layers, device_name=QDEVICE,
)

def count(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'Classical MHA params : {count(classical_mha):>8d}  (embed_dim={embed_dim}, heads={n_heads})')
print(f'Quantum   MHA params : {count(quantum_mha):>8d}  (qubits={n_qubits}, layers={n_layers}, heads={n_heads})')
print(f'Reduction factor     : {count(classical_mha) / max(count(quantum_mha), 1):.2f}x')

In [ ]:
# Smoke-test a forward pass on a tiny token sequence.
tokens = torch.randn(2, 49 + 1, embed_dim)  # (B, N, D), N = patches + CLS
out    = quantum_mha(tokens)
print('QSA forward OK:', tuple(out.shape))

## 3. The four architectures

We instantiate all four models and inspect their parameter counts before training.
* **`ClassicalCNN`** — 3 conv-BN-ReLU-Pool blocks, global average pool, linear head.
* **`vit_tiny`** — embed_dim 192, depth 12, 3 heads (standard `vit_tiny` config).
* **`QCNN`** — Quanvolutional layer (2×2 stride-2 patches → `n_qubits` channels) + small CNN classifier (Henderson et al., 2019).
* **`qvit_tiny`** — embed_dim 96, depth 4, 2 quantum heads — shallower and narrower than vit_tiny to keep simulation tractable, but using QSA blocks throughout.

In [ ]:
from src.models import ClassicalCNN, vit_tiny, QCNN, qvit_tiny

cnn  = ClassicalCNN(num_classes=2)
vit  = vit_tiny(num_classes=2)
qcnn = QCNN(num_classes=2, n_qubits=N_QUBITS, n_layers=2, device_name=QDEVICE)
qvit = qvit_tiny(num_classes=2,
                 n_qubits=N_QUBITS, n_heads=2, n_layers=2,
                 depth=4, embed_dim=96, device_name=QDEVICE)

for name, m in [('CNN', cnn), ('ViT-Tiny', vit), ('QCNN', qcnn), ('QViT', qvit)]:
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:>10s} : {n:>10,d} trainable params')

## 4. Training the classical baselines

The trainer in `src/training/trainer.py` exposes `train_supervised(...)` and `train_distilled(...)`. Both return a `RunHistory` with per-epoch train/val loss and accuracy.

For the notebook we set the epoch budget low so a smoke run fits in a few minutes. Crank `EPOCHS` up to 20-40 for the real benchmark.

In [ ]:
from torch.utils.data import DataLoader
from src.training import train_supervised, train_distilled, history_to_dict

EPOCHS     = 5 if USE_MOCK else 15
BATCH_SIZE = 16

def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0, pin_memory=True)

train_loader = make_loader(train_ds, True)
val_loader   = make_loader(val_ds,   False)
test_loader  = make_loader(test_ds,  False)

hist_cnn = train_supervised(cnn, train_loader, val_loader,
                            epochs=EPOCHS, device=DEVICE, model_name='CNN')

In [ ]:
hist_vit = train_supervised(vit, train_loader, val_loader,
                            epochs=EPOCHS, device=DEVICE, model_name='ViT-Tiny')

## 5. Training the quantum architectures

Quantum forward passes dominate runtime: every token-circuit pair is simulated. On `lightning.gpu` and 4 qubits / 2 layers this is tolerable for the QCNN and a shallow QViT.

Tip: keep `n_qubits ≤ 8` and `depth ≤ 4` for interactive runs. For final results, run `scripts/benchmark.py` as a background job.

In [ ]:
hist_qcnn = train_supervised(qcnn, train_loader, val_loader,
                             epochs=EPOCHS, device=DEVICE, model_name='QCNN')

In [ ]:
hist_qvit = train_supervised(qvit, train_loader, val_loader,
                             epochs=EPOCHS, device=DEVICE, model_name='QViT')

## 6. Knowledge Distillation: ViT-Tiny → QViT

Variational circuits have notoriously rugged loss landscapes (barren plateaus). One practical workaround is **knowledge distillation** (Hinton et al., 2015): the QViT student matches the soft-label distribution of the already-trained classical ViT teacher,

$$\mathcal{L} = \alpha\, \mathrm{CE}(s, y) + (1-\alpha)\, T^2\, \mathrm{KL}\big(\sigma(s/T)\,\Vert\,\sigma(t/T)\big),$$

with temperature $T=4$ and $\alpha=0.5$ by default. This often lets the QViT reach a competitive accuracy with **markedly fewer parameters and epochs** than supervised training from scratch.

In [ ]:
# A fresh QViT student so we don't conflate this run with the supervised one above.
qvit_student = qvit_tiny(num_classes=2,
                         n_qubits=N_QUBITS, n_heads=2, n_layers=2,
                         depth=4, embed_dim=96, device_name=QDEVICE)

hist_qvit_kd = train_distilled(
    student=qvit_student, teacher=vit,
    train_loader=train_loader, val_loader=val_loader,
    epochs=EPOCHS, device=DEVICE,
    temperature=4.0, alpha=0.5,
    model_name='QViT-KD',
)

## 7. Benchmarking — the three required plots

We aggregate per-model results and emit:
* **Accuracy vs. #Trainable Params** — the core efficiency claim.
* **Accuracy vs. Epochs** — convergence behaviour.
* **FLOPs vs. Quantum-Gate Count** — compute footprint (classical FLOPs from forward hooks; quantum gates from an analytical AngleEmbedding + BasicEntanglerLayers count).

In [ ]:
from src.utils import count_flops, plot_acc_vs_params, plot_acc_vs_epochs, plot_flops_vs_gates
from src.models import estimate_gate_count

sample_classical = next(iter(val_loader))['image_classical'][:1].to(DEVICE)

qcnn_gates = estimate_gate_count(n_qubits=N_QUBITS, n_layers=2, n_tokens=(28//2)**2, n_heads=1)['total_gates'] // 3
n_tokens   = (224 // 16) ** 2 + 1
qvit_gates = estimate_gate_count(n_qubits=N_QUBITS, n_layers=2, n_tokens=n_tokens, n_heads=2)['total_gates'] * 4  # depth=4

def record(name, model, hist, qgates=0):
    return {
        'model_name'   : name,
        'n_params'     : hist.n_params,
        'best_val_acc' : max(m.val_acc for m in hist.history),
        'flops'        : count_flops(model.to(DEVICE), sample_classical),
        'quantum_gates': qgates,
    }

records = [
    record('CNN',      cnn,          hist_cnn),
    record('ViT-Tiny', vit,          hist_vit),
    record('QCNN',     qcnn,         hist_qcnn,    qgates=qcnn_gates),
    record('QViT',     qvit,         hist_qvit,    qgates=qvit_gates),
    record('QViT-KD',  qvit_student, hist_qvit_kd, qgates=qvit_gates),
]
histories = [history_to_dict(h) for h in [hist_cnn, hist_vit, hist_qcnn, hist_qvit, hist_qvit_kd]]

print(f"{'model':>10s} | {'params':>10s} | {'best val':>9s} | {'FLOPs':>11s} | {'gates':>8s}")
print('-' * 60)
for r in records:
    print(f"{r['model_name']:>10s} | {r['n_params']:>10,d} | {r['best_val_acc']:>9.3f} | {r['flops']:>11.2e} | {r['quantum_gates']:>8d}")

In [ ]:
import os
os.makedirs('results/plots', exist_ok=True)
plot_acc_vs_params(records,   'results/plots/acc_vs_params.png')
plot_acc_vs_epochs(histories, 'results/plots/acc_vs_epochs.png')
plot_flops_vs_gates(records,  'results/plots/flops_vs_gates.png')

from IPython.display import Image, display
for p in ['acc_vs_params.png', 'acc_vs_epochs.png', 'flops_vs_gates.png']:
    display(Image(f'results/plots/{p}'))

## 8. Analysis & discussion

Three things to inspect in the plots above:

**(a) Parameter efficiency.** On the `Acc vs. Params` chart, the QSA-based models should sit *up-and-to-the-left* relative to the classical baselines. With $n_{qubits}=4, n_{layers}=2, n_{heads}=2, \text{depth}=4$, the QSA blocks contribute only ~`n_layers × n_qubits × 3 (Q/K/V) × n_heads × depth` ≈ 96 quantum-trainable parameters versus ~$10^5$+ for the corresponding classical projections in `vit_tiny`. This is the empirical signature of the $O(n)$ scaling proven in Cherrat et al. (2024).

**(b) Convergence.** On `Acc vs. Epochs`, the **distilled QViT-KD** is the most informative trace: classical-style optimisation hitting variational circuits often stalls early, and KD typically lifts the QViT's first-few-epoch accuracy substantially. If your KD curve is *not* above the from-scratch QViT, try increasing the teacher's training budget or lowering `alpha` to weight the soft labels more.

**(c) Compute footprint.** The `FLOPs vs. Gates` chart highlights the hardware-side tradeoff. Classical models dominate on FLOPs; quantum models trade those FLOPs for gate operations that, on NISQ hardware, may run faster *but* are noisy. The QViT's gate count grows linearly in `n_qubits × n_layers × n_heads × n_tokens × depth`.

### Caveats
* On synthetic-mock data, the QViT's gains will be exaggerated or invisible — the dataset has too little signal. Rerun on the real TyreNet to draw conclusions.
* `lightning.gpu` simulates the quantum circuit; results on real superconducting hardware will incur noise and decoherence not modelled here.
* We freeze the conv reducer used to build `image_quantum`. Letting it learn end-to-end (joint reducer + VQC) is a worthwhile ablation.

### Reproducing the full benchmark

Once you are happy with the wiring, run the full sweep as a script (CLI args mirror the notebook config):

```bash
python scripts/benchmark.py \
    --data_root data/tyrenet \
    --epochs 25 --batch_size 16 \
    --n_qubits 4 --n_layers 2 --n_heads 2 \
    --qvit_depth 4 --qvit_dim 96 \
    --qdevice lightning.gpu --use_kd
```

Outputs land under `results/plots/` and `results/logs/benchmark.json`.

## 9. References

1. Cherrat, El Amine et al. *Quantum Vision Transformers.* (2024).
2. Boucher, P. et al. *Inductive bias of quantum attention.* (2025).
3. Henderson, M. et al. *Quanvolutional Neural Networks.* arXiv:1904.04767 (2019).
4. Hinton, G. et al. *Distilling the Knowledge in a Neural Network.* (2015).
5. Dosovitskiy, A. et al. *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale.* ICLR (2021).